<div style="background: linear-gradient(135deg, rgb(196, 196, 196) 0%, #5a5a5aeb 50%, rgb(0, 0, 0) 100%);
            padding: 10px; border-radius: 8px; margin: 10px 0; color: black; font-family: Arial, sans-serif; text-align: center; font-size: 24px;">

# **Обучения LSTM модели и distilgpt2**
</div>

Ноутбук является пайплайном обучения LSTM модели для задачи автодополнения текста. Модель обучается предсказывать следующие слова в последовательности на основе предыдущего контекста.

### Цель
Создать и обучить языковую модель на основе LSTM архитектуры, которая сможет генерировать осмысленные продолжения для заданных текстовых промптов, и сравнения с исходными данными.

### Основные этапы
- Подготовка данных
- Создание Dataset и DataLoader
- Создание архитектуры LSTM модели
- Обучение модели
- Оценка качества модели при помощи ROUGE-1 и ROUGE-2
- Визуализация  примеров генерации


In [ ]:
# Импорт основных библиотек и модулей
import sys
import os
sys.path.append('..')

import torch
import warnings
from transformers import GPT2TokenizerFast

from src.data_utils import load_config, prepare_data_pipeline
from src.next_token_dataset import NextTokenDataset, get_tokenizer, collate_fn
from src.lstm_model import LSTMLanguageModel
from src.lstm_train import train_lstm_model
from src.eval_transformer_pipeline import TransformerEvaluator

warnings.filterwarnings('ignore')

Для начала выполним инициализацию, настройку окружения и подготовку данных `data_utils.py`
- Проверим доступность GPU
- Выполним загрузку конфигураций из yaml
- Выполним загрузку данных из ранее подготовленного и очищенного датасета
- Разделим данные на train (80%), validation (10%) и test (10%) выборки и сохраним их в csv файлы.


In [2]:
# Настройка устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()

Device: cuda
GPU: NVIDIA GeForce RTX 4070


In [3]:
# Загрузка конфигурации
config = load_config("configs/config.yaml")
print(f"\nПараметры:")
print(f"  max_texts_count: {config['data']['max_texts_count']}")
print(f"  batch_size: {config['training']['batch_size']}")
print(f"  num_epochs: {config['training']['num_epochs']}")

# %%
# Загрузка и разделение данных
train_texts, val_texts, test_texts = prepare_data_pipeline(config, force_reload=True)

# Проверяем, что данные загружены
if len(train_texts) == 0:
    raise ValueError("Не удалось загрузить данные! Проверьте наличие файла data/cleaned_tweets.csv")

Конфигурация загружена configs/config.yaml

Параметры:
  max_texts_count: 1000000
  batch_size: 128
  num_epochs: 10
Принудительная перезагрузка данных

Читаем датасет data/cleaned_tweets.csv...

Всего твитов: 1520716
Используем: 1000000

Размеры выборок:
Train: 800000
Validation: 100000
Test: 100000

Разделенные датасеты сохранены:
data/train.csv
data/val.csv
data/test.csv


Как итог, окружение и конфиг параметры загружены. Прочитан датасет, и использованно 1_000_000 строк из 1_520_716 для модели.
Данные разделены на выборки train val test в соотношении 80% 10% и 10% и сохранены в отдельные датасеты.  

Выполним токенизацию с использованием gpt2 токенизатора

In [4]:
# Загрузка токенизатора
tokenizer = get_tokenizer(config['tokenization']['model_name'])
pad_id = tokenizer.pad_token_id
vocab_size = tokenizer.vocab_size

Создадим PyTorch Dataset и DataLoaderы с формированием батчей для обучения и padding для выравнивания последовательностей разной длины

In [5]:
# Создание датасетов
from torch.utils.data import DataLoader

train_dataset = NextTokenDataset(train_texts, tokenizer, config['tokenization']['max_len'])
val_dataset = NextTokenDataset(val_texts, tokenizer, config['tokenization']['max_len'])
test_dataset = NextTokenDataset(test_texts, tokenizer, config['tokenization']['max_len'])

print(f"Размер train датасета: {len(train_dataset)}")
print(f"Размер val датасета: {len(val_dataset)}")
print(f"Размер test датасета: {len(test_dataset)}")

# Проверяем, что датасеты не пустые
if len(train_dataset) == 0:
    raise ValueError("Train датасет пуст! Возможно, тексты слишком короткие или проблема с токенизацией.")

# %%
# Создание DataLoader'ов
batch_size = config['training']['batch_size']

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_id),
    num_workers=0,
    pin_memory=True if device.type == 'cuda' else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=lambda b: collate_fn(b, pad_id),
    num_workers=0,
    pin_memory=True if device.type == 'cuda' else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=lambda b: collate_fn(b, pad_id),
    num_workers=0,
    pin_memory=True if device.type == 'cuda' else False
)

Размер train датасета: 799897
Размер val датасета: 99985
Размер test датасета: 99986


Создана модель LSTM согласно `lstm_model.py` и обучение `lstm_train.py`
- Использован оптимизатор AdamW
- Настроены оценки ROUGE-1 и ROUGE-2
- Дополнительно выведены PPL Train Loss и Val Loss для более точного анализа.
Кратко
- **Perplexity (PPL)**: метрика уверенности модели в предсказаниях
- **ROUGE-1 и ROUGE-2**: сравнение сгенерированных продолжений с реальными


In [6]:
# Создание модели
model = LSTMLanguageModel(
    vocab_size=vocab_size,
    pad_id=pad_id,
    embed_dim=config['lstm']['embed_dim'],
    hidden_dim=config['lstm']['hidden_dim'],
    num_layers=config['lstm']['num_layers'],
    dropout=config['lstm']['dropout']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Количество параметров: {total_params:,}")

Количество параметров: 29,592,401


In [7]:
# Обучение модели
model = train_lstm_model(model, train_loader, val_loader, tokenizer, device, config)

Эпоха 1/10


Epoch 1 Training: 100%|██████████| 6250/6250 [17:23<00:00,  5.99it/s, loss=4.91]  


Модель сохранена (val_loss: 4.8608)

Результаты эпохи 1:
Train Loss: 5.3481 | PPL: 210.22
Val Loss: 4.8608
ROUGE-1: 0.0781 | ROUGE-2: 0.0122
Время: 1061.78с
Эпоха 2/10


Epoch 2 Training: 100%|██████████| 6250/6250 [14:59<00:00,  6.95it/s, loss=4.36] 


Модель сохранена (val_loss: 4.7029)

Результаты эпохи 2:
Train Loss: 4.9276 | PPL: 138.05
Val Loss: 4.7029
ROUGE-1: 0.0869 | ROUGE-2: 0.0161
Время: 925.69с
Эпоха 3/10


...

Модель сохранена (val_loss: 4.5022)

Результаты эпохи 8:
Train Loss: 4.6475 | PPL: 104.32
Val Loss: 4.5022
ROUGE-1: 0.0788 | ROUGE-2: 0.0098
Время: 816.49с
Эпоха 9/10


Epoch 9 Training: 100%|██████████| 6250/6250 [15:22<00:00,  6.78it/s, loss=4.77]


Модель сохранена (val_loss: 4.4927)

Результаты эпохи 9:
Train Loss: 4.6336 | PPL: 102.89
Val Loss: 4.4927
ROUGE-1: 0.0828 | ROUGE-2: 0.0110
Время: 939.59с
Эпоха 10/10


Epoch 10 Training: 100%|██████████| 6250/6250 [14:46<00:00,  7.05it/s, loss=5.05]


Модель сохранена (val_loss: 4.4819)

Результаты эпохи 10:
Train Loss: 4.6226 | PPL: 101.75
Val Loss: 4.4819
ROUGE-1: 0.0806 | ROUGE-2: 0.0170
Время: 936.39с


В процессе обучения мы видим стабильное снижение loss, а также ускорения обучения (сокращение обработки эпох и увеличения скорости обработки) - возможно вначале был некий разогрев модели.

Лучший результат ROUGE-1 - 0.0938 (эпоха 7)
Лучший результат ROUGE-2 - 0.0170 (эпоха 10)
Но результаты метрик ROUGE - оставляют желать лучшего, возможно из-за качества данных, или необходимо дополнительная настройка штрафов за повторения или настройка длины генерации.

Что касается PPL - снижается плавно при следующей эпохе, это значит, модель учится. Для улучшения PPL можно увеличить количество эпох и значения выйдет ниже 100.

Тестирование LSTM модели `eval_lstm.py`

In [8]:
# Тестирование на примерах
from src.eval_lstm import test_lstm_examples
test_lstm_examples(model, tokenizer, device)


Промпт: Hello, my name is
Генерация: Hello, my name is the coolest! i wish i had a new video! i'll be there

Промпт: Today is a beautiful
Генерация: Today is a beautiful one. but i'm missing a bbq in the sun. i

Промпт: How are you doing
Генерация: How are you doing a good job? i think it's all over. i wish i could

Промпт: I love to
Генерация: I love to you too. good luck with it. we're really glad i was too

Промпт: Machine learning is
Генерация: Machine learning is not good! i feel like a big idiot. i don't know what


После тестирования модели мы видим резкие переходы в тексте, несвязность и нарушения структуры текста, а также часто повторящие конструкции и применения `i`.

Мне кажется это проблемы данных, необходимо добавить качественные данные а именно новости или какие литературные тексты, чтобы модель могла обучиться на них а потом при использование контекста твитов обучилась стилистикой.

Как итог модель научилась генерировать англоязычный текст, но пока с потерей контекста, эмоциональными скачками, незаконченностью.

In [9]:
# %%
# Сравнение с трансформером
print("DistilGPT2")

transformer_eval = TransformerEvaluator("distilgpt2")

# Оценка ROUGE для LSTM
from src.eval_lstm import evaluate_rouge_lstm
rouge1, rouge2 = evaluate_rouge_lstm(
    model, val_loader, tokenizer, device, 
    num_examples=config['evaluation']['rouge_examples']
)

# Оценка ROUGE для трансформера
gpt2_rouge1, gpt2_rouge2 = transformer_eval.evaluate_rouge(
    val_texts, 
    num_examples=config['evaluation']['rouge_examples'],
    split_ratio=0.75
)

print(f"\nROUGE метрики:")
print(f"LSTM - ROUGE-1: {rouge1:.4f}, ROUGE-2: {rouge2:.4f}")
print(f"GPT2 - ROUGE-1: {gpt2_rouge1:.4f}, ROUGE-2: {gpt2_rouge2:.4f}")

# %%

DistilGPT2


Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



ROUGE метрики:
LSTM - ROUGE-1: 0.0938, ROUGE-2: 0.0173
GPT2 - ROUGE-1: 0.0568, ROUGE-2: 0.0000


<div style="font-family: 'Courier New', monospace; font-size: 120%; background: linear-gradient(135deg, rgb(126, 126, 126) 0%, #f4f4f4 30%, #ffffff 60%, #000000 100%); border-left: 4px solid #ffffff; padding: 12px; margin: 10px 0; border-radius: 8px; color: rgb(0, 0, 0); box-shadow: 0 0 15px rgba(71, 255, 237, 0.6), 0 0 20px rgba(255, 71, 87, 0.2); text-shadow: 0 1px 1px rgba(0, 0, 0, 0.4);">
<strong style="color: rgb(0, 0, 0); font-weight: 600;">Основные моменты</strong>
</div>



Выполнена полная подготовка данных, разделены и сохраненны датасеты, созданы даталоадеры и архитектура LSTM модели. Проведен анализ Оценки качества через ROUGE-1 и ROUGE-2 и приведена визуализация примеров генерации LSTM.

Исходя из метрик можно сказать, что LSTM обогнала DistilGPT2 по ROUGE метрикам! Но для более качественного анализа необходимо перейти в `notebook_test_models.ipynb` где будет проходить непосредственно тестирование полученных моделей.